# 46 — Pre-production evidence

Synthesizes M5 evidence into a robustness card and monitoring contract. Passing this notebook **does not authorize production**.


In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.robustness import load_robustness_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
(paths.root / "reports" / "robustness").mkdir(parents=True, exist_ok=True)
(paths.root / "data" / "m5").mkdir(parents=True, exist_ok=True)
(paths.root / "artifacts" / "models" / "m5").mkdir(parents=True, exist_ok=True)

m4_gate = paths.docs / "decisions" / "M4_GATE.md"
assert m4_gate.exists() and "M4 ✅ CLOSED" in m4_gate.read_text(encoding="utf-8"), (
    "M4 no está formalmente cerrado. No iniciar M5."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M5_SPEC = load_robustness_spec(paths.configs / "robustness.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

assert GOLD_SPEC.schema_version == M5_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M5_SPEC.core.benchmark_model_schema
{
    "gold_schema": GOLD_SPEC.schema_version,
    "model_schema": MODEL_SPEC.schema_version,
    "m5_schema": M5_SPEC.schema_version,
    "benchmark_model": M5_SPEC.core.benchmark_model_name,
    "benchmark_calibration": M5_SPEC.core.benchmark_calibration,
    "consumed_test_year": M5_SPEC.governance.consumed_test_year,
    "fresh_oot_year": M5_SPEC.governance.fresh_oot_year,
}

import pandas as pd
path45 = paths.reports / "robustness" / "45_validation.parquet"
assert path45.exists(), "Ejecuta 45 primero."
check45 = pd.read_parquet(path45)
assert check45.loc[check45["blocking"], "passed"].all(), "45 no está aprobado."
from argos.robustness.readiness import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm5-46-preproduction-evidence-v1'

In [2]:
from argos.robustness.readiness import build_preproduction_evidence

result = build_preproduction_evidence(
    root=paths.root,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
    docs_root=paths.docs,
    modeling_spec=MODEL_SPEC,
    robustness_spec=M5_SPEC,
)
result.evidence_summary


,evidence,value,interpretation
0,rolling_origin_folds,4,development-only temporal robustness
1,global_features_explained,41,permutation importance coverage
2,uncertainty_metrics,7,block-bootstrap intervals
3,high_drift_pairs,3,"monitor, do not silently ignore"
4,competition_increment_folds,3,fair incremental feature-family comparisons
5,fresh_oot_rows,529054,post-2025 fully observed partial-year holdout


In [3]:
result.monitoring_thresholds


,signal,warning,high,action
0,numeric_feature_psi,0.10,0.25,investigate feature distribution / upstream da...
1,categorical_feature_tvd,0.10,0.25,investigate category mix and unseen values
2,score_psi,0.10,0.25,review score shift and calibration
3,prevalence_relative_change,0.30,NaN,review prior probability shift; recalibration ...
4,tail_calibration_abs_gap,0.03,NaN,review top-score calibration before operationa...
5,lift_at_operational_k,1.00,NaN,pause promotion if lift is at/below random review
6,feature_missingness_absolute_delta,0.05,NaN,investigate ingestion/schema/source changes


In [4]:
result.validation


,check_id,category,blocking,passed,observed,requirement
0,m5_component_evidence_complete,evidence,True,True,"{""40_validation.parquet"": {""blocking"": 4, ""fai...",notebooks 40–45 have all blocking evidence che...
1,monitoring_contract_persisted,monitoring,True,True,"{""path"": ""/Users/macm1pro/Desktop/projects/arg...",pre-production monitoring thresholds and deter...
2,robustness_card_persisted,documentation,True,True,"{""path"": ""/Users/macm1pro/Desktop/projects/arg...",ARGOS V1 has a dedicated robustness card disti...
3,fresh_oot_consumed_without_reselection,governance,True,True,"{""fresh_oot_consumed"": true, ""model_reselected...",2026 holdout is recorded as consumed and does ...
4,model_runtime_pinned,portability,True,True,"{""scikit_learn_uv_lock"": ""1.9.1""}",serialized M4 model runtime remains pinned to ...
5,frozen_model_lineage_intact,lineage,True,True,"{""calibrator_sha256"": ""95e8e506f92b191f7315a1f...",all M5 final holdout evidence refers to the ex...
6,production_not_auto_authorized,governance,True,True,"{""production_authorized"": false}",M5 evidence completion must not silently autho...


In [5]:
blocking = result.validation.loc[result.validation["blocking"]]
print("46 ✅ PASS" if result.ready_for_next_notebook else "46 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Robustness card:", result.robustness_card_path)
print("Monitoring spec:", result.monitoring_spec_path)
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook


46 ✅ PASS
Blocking checks: 7/7
Robustness card: /Users/macm1pro/Desktop/projects/argos/argos-code/docs/models/ARGOS_V1_ROBUSTNESS_CARD.md
Monitoring spec: /Users/macm1pro/Desktop/projects/argos/argos-code/docs/monitoring/ARGOS_V1_MONITORING_SPEC.md
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m5_preproduction_evidence.json
